# Debugging Pandas

### Loading Libraries

In [ ]:
# Numerical Computing
import numpy as np

# Data Manipualtion
import pandas as pd

# Data Visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# PyArrow
import pyarrow as pa

# URL
import urllib.request

In [ ]:
with open('/Users/joaquinromero/Desktop/EP2/data/devilclean.txt', 'r') as f:
    for i in range(10):
        print(repr(f.readline()))

In [ ]:
url = (
    'https://github.com/mattharrison/datasets/raw/master'
    '/data/dirtydevil.txt'
)

local_filename = '/Users/joaquinromero/Desktop/EP2/data/devilclean.txt'

urllib.request.urlretrieve(url, local_filename)

In [ ]:
with open(local_filename, 'r') as file:
    lines = file.readlines()

with open(local_filename, 'w') as file:
    for i, line in enumerate(lines):

        # elimina metadata del USGS
        if i < 34 or i == 35:
            continue

        file.write(line)

In [ ]:
with open(local_filename, 'r') as f:
    for i in range(5):
        print(repr(f.readline()))

In [ ]:
def to_denver_time(df_, time_col, tz_col):

    return (

        df_

        .assign(**{tz_col: df_[tz_col].replace('MDT', 'MST7MDT')})

        .groupby(tz_col)[time_col]

        .transform(

            lambda s:

                pd.to_datetime(s)

                .dt.tz_localize(s.name, ambiguous=True)

                .dt.tz_convert('America/Denver')

        )

    )

def tweak_river(df_):

    return (

        df_

        .assign(

            datetime=to_denver_time(df_, 'datetime', 'tz_cd')

        )

        .rename(

            columns={

                '144166_00060': 'cfs',

                '144167_00065': 'gage_height'

            }

        )

        .loc[

            :,

            [

                'datetime',

                'agency_cd',

                'site_no',

                'tz_cd',

                'cfs',

                'gage_height'

            ]

        ]

    )

#### Checking if DataFrames are Equal

In [ ]:
df = pd.read_csv('/Users/joaquinromero/Desktop/EP2/data/devilclean.txt', sep='\t', dtype_backend='pyarrow')

In [ ]:
dd = tweak_river(df)

print(dd)

In [ ]:
dd2 = pd.read_json(
    dd.to_json(),
    dtype_backend='pyarrow'
)

dd.equals(dd2)

In [ ]:
print(dd.eq(dd2))

In [ ]:
(dd
    .eq(dd2)
    .sum()
)

In [ ]:
pd.testing.assert_frame_equal(dd, dd2)

In [ ]:
from_json = (dd2
    .assign(datetime=dd2.datetime
        .dt.tz_localize('UTC')
        .dt.tz_convert('America/Denver'))
            )

In [ ]:
pd.testing.assert_frame_equal(dd, from_json)

In [ ]:
pd.testing.assert_frame_equal(
    dd,
    from_json,
    check_dtype=False
)

In [ ]:
dd.equals(from_json)

In [ ]:
pd.testing.assert_frame_equal(
    dd,
    from_json.astype(dict(dd.dtypes))
)

In [ ]:
dd.equals(from_json.astype(dict(dd.dtypes)))

In [ ]:
pd.testing.assert_frame_equal(
    dd,
    from_json.astype(dict(dd.dtypes)),
    check_exact=True
)

In [ ]:
dd3 = from_json.astype(dict(dd.dtypes))

In [ ]:
dd.eq(dd3).all()

In [ ]:
print(dd[dd.cfs.ne(dd3.cfs)])

In [ ]:
dd.loc[96246].cfs, dd3.loc[96246].cfs

In [ ]:
def cmp_dfs(df1, df2, round_amt=3):

    diff_cols = set(df1.columns) ^ set(df2.columns)

    if diff_cols:
        print(f'Different columns {diff_cols}')

    if df1.shape != df2.shape:
        print(f'Different shapes {df1.shape} {df2.shape}')

    bad = False

    for col in df1.columns:

        s1 = df1[col]
        s2 = df2[col]

        if s1.equals(s2):
            continue

        bad = True

        if s1.dtype != s2.dtype:

            print(
                f'{col} types differ {s1.dtype} vs {s2.dtype}'
            )

        if s1.dtype in ['float', 'double[pyarrow]']:

            if s1.round(round_amt).equals(
                s2.round(round_amt)
            ):

                print(
                    f'{col} has rounding differences'
                    f'{df1[s1.ne(s2)][col].dropna().iloc[0]} '
                    f'vs '
                    f'{df2[s1.ne(s2)][col].dropna().iloc[0]}'
                )

        else:

            diff = (
                df1
                .loc[s1.ne(s2)]
                .assign(other=s2)
                .loc[:, [col, 'other']]
                .dropna()
            )

            print(f'{col} differs {diff}')

    if not bad:
        print('Same')

In [ ]:
cmp_dfs(dd, dd3)

In [ ]:
#### Debugging Chains